# Dự đoán giá HDB Resale Singapore — v2 (evaluation sạch)

**Khác biệt cốt lõi so với v1:** thay random split bằng **time-based split**; test 2024 được giữ
*untouched* cho đến cell đánh giá cuối cùng. Mọi quyết định (feature, transform, hyperparameter,
model family) đều được chọn trên **validation 2023** hoặc year-based CV trong tập train.

| Tập | Năm | Số mẫu |
|---|---|---|
| Train | 2015–2022 | 178,587 |
| Validation | 2023 | 25,478 |
| Test | 2024 | 16,906 *(không đụng đến trước cell Final)* |

**Kết quả cuối (test 2024, LightGBM):** R² **0.9008** · RMSE **$58,478** · MAE **$41,180** ·
46.99% dự đoán trong ±5% · **79.46% trong ±10%**.

Các thí nghiệm ablation / target-transform / tuning được chạy bằng script trong `v2_experiments/`
(kết quả load từ CSV ở đây để notebook chạy nhanh; script có trong repo nên tái lập được 100%).

## 1. Load dữ liệu & audit

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('resale.csv')
print(f"Số dòng: {len(df):,} | Cột: {list(df.columns)}")
print(f"Trùng lặp hoàn toàn: {df.duplicated().sum()} | Thiếu giá trị: {df.isnull().sum().sum()}")

by_year = df.groupby('year')['resale_price'].agg(['size', 'median'])
by_year.columns = ['n_giao_dịch', 'median_giá']
by_year

Số dòng: 220,971 | Cột: ['year', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'remaining_lease_years', 'resale_price', 'storey_range_category', 'distance_from_expressway']


Trùng lặp hoàn toàn: 0 | Thiếu giá trị: 0


,n_giao_dịch,median_giá
year,,
2015,17642,405000.0
2016,19186,410000.0
2017,20157,410000.0
2018,21345,408000.0
2019,21917,400000.0
2020,23052,425000.0
2021,28772,483000.0
2022,26516,525000.0
2023,25478,550000.0


**Audit — kết luận chính:**
- Median giá tăng $400–410k (2015–2019) → $580k (2024): dữ liệu **không stationary**. Random split
  sẽ cho mô hình nhìn thấy phân phối giá của tương lai → chỉ số đánh giá lạc quan ảo. Đó là lỗi
  phương pháp chính của v1.
- Không có null, không trùng dòng. Tất cả feature đều biết được tại thời điểm niêm yết (point-in-time).
- `storey_range` (17 giá trị, format `XX TO YY`) có tín hiệu giá đơn điệu mạnh: $413k (tầng 1–3)
  → $1.11M (tầng 49–51). `distance_from_expressway` tín hiệu yếu, 86% là ">500m".

## 2. Feature engineering & time-based split

In [2]:
# distance_from_expressway: 6 bậc có thứ tự -> numeric theo midpoint
DIST_MAP = {'<=50m': 25, '51-100m': 75, '101-150m': 125, '151-300m': 225, '301-500m': 400, '>500m': 500}
df['distance_ord'] = df['distance_from_expressway'].map(DIST_MAP)

# storey_range '04 TO 06' -> điểm giữa tầng 5.0
sr = df['storey_range'].str.extract(r'(\d+)\s*TO\s*(\d+)')
df['storey_mid'] = (sr[0].astype(int) + sr[1].astype(int)) / 2

NUM = ['year', 'floor_area_sqm', 'remaining_lease_years', 'storey_mid']
CAT = ['town', 'flat_type']
FEATURES = NUM + CAT

train = df[df['year'] <= 2022]   # tuning + fitting
val   = df[df['year'] == 2023]   # mọi so sánh model/feature
test  = df[df['year'] == 2024]   # KHÔNG đụng đến trước cell cuối
print(f"Train {len(train):,} | Val {len(val):,} | Test {len(test):,}")

Train 178,587 | Val 25,478 | Test 16,906


**Feature selection có bằng chứng (ablation, LightGBM, đánh giá trên val 2023):**

| Cấu hình | R² | RMSE | ±10% |
|---|---|---|---|
| base (5 features, không storey/distance) | 0.8501 | $67,357 | 70.4% |
| base + distance | 0.8520 | $66,933 | 70.6% |
| base + storey (one-hot) | 0.8653 | $63,836 | 73.6% |
| **base + storey_mid** | **0.8686** | **$63,067** | **73.7%** |
| base + dist + storey_mid | 0.8671 | $63,418 | 73.8% |

→ `storey_mid` giảm RMSE $4,290; mid-floor thắng one-hot; **distance bị loại** (vô dụng khi đã có
storey — trả lời definitively câu hỏi v1 bỏ ngỏ). Negative result khác: **log1p(target) tệ hơn raw**
trên mọi metric với cả LightGBM lẫn RF → giữ raw target (chi tiết: `v2_experiments/phase4_target.csv`).

## 3. Baselines (đánh giá trên val 2023)

In [3]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def within_pct(y_true, y_pred, tol):
    return float(((y_pred >= y_true * (1 - tol)) & (y_pred <= y_true * (1 + tol))).mean() * 100)

def evaluate(y_true, y_pred):
    return {'R2': r2_score(y_true, y_pred),
            'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'MAE': float(mean_absolute_error(y_true, y_pred)),
            'Pct_within_10pct': within_pct(y_true, y_pred, 0.10)}

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]).set_output(transform='pandas'), NUM),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]).set_output(transform='pandas'), CAT),
])

X_val = pre.fit(train[FEATURES]).transform(val[FEATURES])
y_val = val['resale_price'].values
X_train_all = pre.transform(train[FEATURES])
y_train = train['resale_price'].values

baseline_rows = []
for name, model in [
    ('Median', DummyRegressor(strategy='median')),
    ('Ridge', Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1.0))])),
    ('RandomForest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ('LightGBM', lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)),
]:
    model.fit(X_train_all, y_train)
    r = evaluate(y_val, model.predict(X_val)); r['Model'] = name
    baseline_rows.append(r)
    print(f"{name:14s} R2={r['R2']:.4f} RMSE={r['RMSE']:>10,.0f} MAE={r['MAE']:>9,.0f} ±10%={r['Pct_within_10pct']:5.1f}%")

Median         R2=-0.6248 RMSE=   221,742 MAE=  167,141 ±10%= 17.2%


Ridge          R2=0.7170 RMSE=    92,544 MAE=   71,712 ±10%= 44.3%


RandomForest   R2=0.8765 RMSE=    61,139 MAE=   44,519 ±10%= 73.5%


LightGBM       R2=0.8686 RMSE=    63,067 MAE=   44,609 ±10%= 73.7%


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Baseline Median cho R² âm (−0.62) vì median train ($435k) lệch hẳn median val 2023 ($550k) —
minh chứng trực tiếp cho mức dịch phân phối theo thời gian. LightGBM default ngang RF nhưng nhanh
hơn ~300 lần → dùng LightGBM cho tuning.

In [4]:
pd.DataFrame(baseline_rows)[['Model','R2','RMSE','MAE','Pct_within_10pct']]

,Model,R2,RMSE,MAE,Pct_within_10pct
0,Median,-0.624798,221742.105264,167140.579546,17.210927
1,Ridge,0.716993,92543.705763,71711.593604,44.304890
2,RandomForest,0.876480,61138.821955,44518.834486,73.510480
3,LightGBM,0.868566,63067.015209,44609.329212,73.663553


## 4. Tuning — year-based CV, không đụng val/test
Folds: (2015–2019→2020), (2015–2020→2021), (2015–2021→2022). LightGBM 20 trials, RF 12 trials
(có `max_features`). Kết quả đầy đủ: `v2_experiments/phase5_tuning_summary.csv`.

In [5]:
tuning = pd.read_csv('v2_experiments/phase5_tuning_summary.csv')
tuning[['Family', 'CV_R2_mean', 'CV_R2_std', 'Val2023_R2', 'Val2023_RMSE', 'Val2023_MAE']]

,Family,CV_R2_mean,CV_R2_std,Val2023_R2,Val2023_RMSE,Val2023_MAE
0,LightGBM,0.855565,0.038606,0.888822,58003.956865,42266.536144
1,RandomForest,0.850288,0.038106,0.883056,59489.120029,42759.165016


LightGBM thắng cả CV (0.8556 vs 0.8503) lẫn val 2023 (0.8888 vs 0.8831) → chốt LightGBM với
params: `n_estimators=400, learning_rate=0.05, num_leaves=127, min_child_samples=100,
colsample_bytree=0.7, subsample=1.0`. RF vẫn được giữ làm sanity-check ở Phase 2/4.

## 5. FINAL — train 2015–2023, đánh giá duy nhất một lần trên test 2024

In [6]:
BEST_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=127,
                   min_child_samples=100, colsample_bytree=0.7, subsample=1.0, subsample_freq=1)

fit_df = df[df['year'] <= 2023]
X_fit = pre.fit_transform(fit_df[FEATURES])
y_fit = fit_df['resale_price'].values
X_test = pre.transform(test[FEATURES])
y_test = test['resale_price'].values

final_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **BEST_PARAMS)
final_model.fit(X_fit, y_fit)
y_pred = final_model.predict(X_test)

final_metrics = evaluate(y_test, y_pred)
print("=== Test 2024 (chưa từng bị đụng đến) ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v:,.4f}")

=== Test 2024 (chưa từng bị đụng đến) ===
  R2: 0.9008
  RMSE: 58,477.8214
  MAE: 41,180.4159
  Pct_within_10pct: 79.4629


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**So sánh với v1:** v1 báo test R² 0.9452 / ±10% 84.05% — nhưng từ random split, nơi tập test
chứa đúng phân phối giá của các năm đã học. Con số này của v2 (R² 0.9008 trên **năm hoàn toàn
chưa thấy**) mới là ước lượng đáng tin cho câu hỏi "mô hình dự đoán năm tới tốt đến đâu".
R² giảm không phải vì model yếu hơn — LightGBM ở đây mạnh hơn RF của v1 — mà vì cách đo trung thực hơn.

## 6. Phân tích lỗi

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

preds = test[['year', 'town', 'flat_type', 'storey_range', 'floor_area_sqm',
              'remaining_lease_years', 'resale_price']].copy()
preds['predicted'] = y_pred
resid = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].scatter(y_test, y_pred, alpha=0.3, s=6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set(xlabel='Actual (SGD)', ylabel='Predicted (SGD)', title='Actual vs Predicted — test 2024')
axes[1].scatter(y_pred, resid, alpha=0.3, s=6)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set(xlabel='Predicted (SGD)', ylabel='Residual (SGD)', title='Residuals vs Predicted — test 2024')
plt.tight_layout(); plt.savefig('v2_analysis_charts.png', dpi=110); plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_1472\361997001.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('v2_analysis_charts.png', dpi=110); plt.show()


In [8]:
segments = [(0, 300_000, '<$300k'), (300_000, 500_000, '$300–500k'), (500_000, 700_000, '$500–700k'),
            (700_000, 1_000_000, '$700k–1M'), (1_000_000, np.inf, '>$1M')]
rows = []
for lo, hi, name in segments:
    m = (y_test >= lo) & (y_test < hi)
    rows.append({'Segment': name, 'n': int(m.sum()),
                 'MAE': float(np.abs(resid[m]).mean()),
                 'Pct_within_10pct': within_pct(y_test[m], y_pred[m], 0.10)})
seg = pd.DataFrame(rows)
seg

,Segment,n,MAE,Pct_within_10pct
0,<$300k,123,16155.088065,81.300813
1,$300–500k,5094,22079.838212,86.808009
2,$500–700k,7381,34799.142769,83.308495
3,$700k–1M,3765,65319.212487,68.472776
4,>$1M,543,145405.215229,34.069982


**Phát hiện then chốt:** MAE tăng gần tuyến tính theo phân khúc giá và phân khúc >$1M chỉ có
34.1% dự đoán trong ±10% (so với 86.8% ở $300–500k). RMSE trung bình của toàn tập đang che giấu
việc model kém nhất ở đúng nơi giá trị tiền lớn nhất. Hướng cải thiện tiếp theo: feature khu vi mô
(đã có `street_name` — thử target encoding), hoặc model riêng cho phân khúc cao / quantile loss.
Log-transform **không** phải giải pháp — đã thử và tệ hơn (Phase 4).

In [9]:
# Permutation importance ở mức feature (ΔRMSE khi xáo trộn; test 2024, 5 repeats)
rng = np.random.default_rng(42)
test_feat = test[FEATURES].reset_index(drop=True)
base_rmse = final_metrics['RMSE']
imp = []
for feat in FEATURES:
    deltas = []
    for _ in range(5):
        sh = test_feat.copy()
        sh[feat] = rng.permutation(sh[feat].values)
        p = final_model.predict(pre.transform(sh))
        deltas.append(np.sqrt(((y_test - p) ** 2).mean()) - base_rmse)
    imp.append({'Feature': feat, 'dRMSE': float(np.mean(deltas)), 'std': float(np.std(deltas))})
pd.DataFrame(imp).sort_values('dRMSE', ascending=False)

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Feature,dRMSE,std
1,floor_area_sqm,109714.873928,994.132461
4,town,78926.787322,616.883156
2,remaining_lease_years,65614.365215,310.913591
5,flat_type,24203.264338,408.709862
3,storey_mid,16976.026031,119.945203
0,year,0.000000,0.000000


Lưu ý: `year` có ΔRMSE = 0 — vì test 2024 chỉ chứa đúng một giá trị năm, xáo trộn không đổi gì.
Vai trò của `year` (proxy xu hướng giá) nằm trong quá trình học, không đo được trên test 1 năm.

## 7. Tổng kết cho phỏng vấn
1. **Vấn đề của v1:** random split trên dữ liệu thời gian (không stationary, median +33% từ train
   sang test) → R² 0.945 lạc quan. Tuning với search space hẹp bao quanh giá trị đã biết.
2. **Sửa:** time-based split (train ≤2022, val 2023, test 2024 untouched); year-based CV cho tuning;
   mọi so sánh trên cùng một validation.
3. **Giá trị tạo ra:** ablation chứng minh `storey` đáng $4.3k RMSE và `distance` vô dụng;
   negative result về log-transform; LightGBM thay RF (nhanh ~300×, chính xác hơn).
4. **Kết quả trung thực:** R² 0.9008, ±10% 79.5% trên năm chưa thấy — và biết rõ model yếu ở đâu
   (phân khúc >$1M: 34.1% ±10%; các town trung tâm: ~58–63%).
5. **Bước tiếp theo:** target encoding `street_name`, quantile loss / model riêng cho segment cao.

## 8. Nâng cấp v3: street target-encoding + khoảng dự đoán conformal

Hai nâng cấp khả thi hoàn toàn với dataset hiện có, giữ nguyên protocol time-split:
quyết định chốt trên **val 2023**, test 2024 chỉ mở ở cuối (chi tiết script: `v2_experiments/phase9–11`).

### 8.1. Target encoding `street_name` (566 giá trị) — 3 lớp chống leakage
Fit encoding **chỉ trên train**, smoothing `(n·mean + m·prior)/(n + m)` với `m=10` (thí nghiệm m∈{10,50,200}),
street lạ → fallback town-level (val 2023 chỉ 12/25,478 dòng rơi vào trường hợp này).

In [10]:
te_results = pd.read_csv('v2_experiments/phase9_street_te.csv')
te_results

,Config,R2,RMSE,MAE,Pct_within_10pct
0,"control (6 features, no TE)",0.888822,58003.956865,42266.536144,76.332522
1,"A: raw-price TE, m=10",0.907186,52997.588901,39773.929694,79.115315
2,"A: raw-price TE, m=50",0.906258,53261.777320,39962.206830,78.828793
3,"A: raw-price TE, m=200",0.905751,53405.690972,40400.378531,78.338174
4,"B: premium TE (point-in-time), m=10",0.906296,53250.847179,40132.155513,78.832718
5,"B: premium TE (point-in-time), m=50",0.904579,53736.765720,40384.329927,78.499097
6,"B: premium TE (point-in-time), m=200",0.906795,53108.889633,40205.048474,78.353874


**Kết quả (val 2023):** raw-price TE m=10 giảm RMSE **$58,004 → $52,998 (−$5,006)**, R² 0.8888 → 0.9072.
Biến thể "premium index" chuẩn hóa point-in-time cho kết quả tương đương (0.9063–0.9068) → chọn
biến thể đơn giản hơn. So với Phase 3, đây là feature đơn lẻ mạnh nhất kể từ `storey_mid`.

### 8.2. Khoảng dự đoán: quantile thuần thất bại → conformal phân đoạn
**Negative result đáng nhớ:** LGBM quantile (α=0.1/0.5/0.9) chỉ phủ **53.5%** (kỳ vọng 80%), segment
>$1M chỉ 16.9% — một model quantile toàn cục không nắm được heteroscedasticity theo phân khúc.

Giải pháp: **relative conformal calibration** — score `|resid|/pred` calibrate trên năm 2023
(model fit ≤2022, gap 1 năm), width theo nhóm giá dự đoán, nhóm thưa fallback nhóm gần nhất.

In [11]:
q_seg = pd.read_csv('v2_experiments/phase10_quantile_segments.csv')
print('Quantile thuần (val 2023) — coverage mục tiêu ~80%:')
print(q_seg.to_string(index=False))
v3_seg = pd.read_csv('v2_experiments/phase11_conformal_segments_v3.csv')
print('\nConformal relative (test 2024) — coverage mục tiêu 90%:')
print(v3_seg.to_string(index=False))

Quantile thuần (val 2023) — coverage mục tiêu ~80%:
  Segment     n  Coverage_pct    Mean_width       MAE_q50
   <$300k   324     79.938272  49928.700737  16440.311736
$300–500k  8975     59.899721  52916.198910  23997.947049
$500–700k 10934     52.076093  77701.465224  39543.421128
 $700k–1M  4777     46.598283 119940.752351  68705.814023
     >$1M   468     16.880342 180420.323619 150775.180073

Conformal relative (test 2024) — coverage mục tiêu 90%:
Segment (giá thật)    n  Coverage_pct    Mean_width  MAE_q50_point
            <$300k  123     91.869919  37835.542200   15324.736045
         $300–500k 5094     95.131527  53575.360823   20486.243308
         $500–700k 7381     95.163257  83562.027689   32708.221077
          $700k–1M 3765     89.110226 116423.110216   56098.464312
              >$1M  543     66.850829 151638.159095  128205.494148


### 8.3. FINAL v3 — test 2024 mở duy nhất một lần

In [12]:
v3_metrics = pd.read_csv('v2_experiments/phase11_final_metrics_v3.csv')
print('=== FINAL v3 (LightGBM + street TE, train 2015–2023) — test 2024 ===')
print(v3_metrics.T.rename(columns={0: 'value'}))
compare = pd.DataFrame({
    'Phiên bản': ['v1 (random split — lạc quan)', 'v2 (time-split)', 'v3 (time-split + street TE)'],
    'R2': [0.9452, 0.9008, 0.9217],
    'RMSE': [39955, 58478, 51963],
    'MAE': [28296, 41180, 37175],
    'Pct_within_10pct': [84.05, 79.46, 84.25],
})
compare

=== FINAL v3 (LightGBM + street TE, train 2015–2023) — test 2024 ===
                         value
R2                    0.921677
RMSE              51963.027386
MAE               37175.413887
Pct_within_5pct      49.662842
Pct_within_10pct     84.254111
n_test            16906.000000


,Phiên bản,R2,RMSE,MAE,Pct_within_10pct
0,v1 (random split — lạc quan),0.9452,39955,28296,84.05
1,v2 (time-split),0.9008,58478,41180,79.46
2,v3 (time-split + street TE),0.9217,51963,37175,84.25


**Đọc kết quả:** v3 đạt **R² 0.9217 / ±10% 84.25%** trên năm chưa từng thấy — gần như san bằng
con số 0.945 "ảo" của v1 nhưng với phương pháp sạch. Coverage conformal toàn tập **92.9%**
(mục tiêu 90%).

**Hạn chế báo cáo trung thực:** segment >$1M chỉ được phủ 66.9% — nhóm này có 0 mẫu >$1M trong
năm calibration 2023 (fallback width từ nhóm $700k–1M). Conformal đảm bảo coverage *biên* (marginal);
coverage *theo phân khúc* cần thêm dữ liệu calibration cho nhóm cao. Đây là limitation chính thức
của v3 và là bước tiếp theo rõ ràng nhất.